In [1]:
import torch
import torch.nn as nn
import numpy as np
import torch.optim as optim
import pandas as pd
from transformers import AutoModel, AutoTokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df = pd.read_csv('merged_with_tweets_training.csv')
cols_to_clean = ["Temperature", "Humidity", "Pressure", "Visibility", "Wind"]

for col in cols_to_clean:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"[^\d\.\-]", "", regex=True)
        .replace("", None)
        .astype(float)
    )
df = df.fillna(0)
freq = df["Condition"].value_counts(normalize=True)
df["Condition"] = df["Condition"].map(freq)
df

,Unnamed: 0,timestamp,geometry,tweets,Boro,Vol,congestion,congestion_level,Temperature,Condition,Wind,Humidity,Pressure,Visibility
0,0,2025-02-22 03:30:00,POLYGON ((-73.98203507773665 40.65524805627688...,"['Just drove to the store and it was a breeze,...",Brooklyn,8,-1.589893,0,-6.0,0.150429,9.0,54.0,1029.0,16.0
1,1,2024-11-09 09:15:00,POLYGON ((-73.93291959036472 40.82741450154307...,"[""Just headed out for a run and it's a beautif...",Manhattan,264,0.560178,0,7.0,0.329696,0.0,40.0,1027.0,16.0
2,2,2025-03-06 13:45:00,POLYGON ((-73.97178007473232 40.79434366943585...,"[""Just braved the chilly morning commute, it's...",Manhattan,123,0.635957,0,11.0,0.127825,0.0,50.0,992.0,16.0
3,3,2024-04-04 15:30:00,POLYGON ((-73.98780645780917 40.67354750428635...,"['Woke up to a pretty chilly grey day, 9°C out...",Brooklyn,151,0.744760,0,9.0,0.127825,6.0,52.0,996.0,16.0
4,4,2025-01-08 21:15:00,POLYGON ((-73.91101950432183 40.87001883438244...,"['Just got out of my car, finally parked in a ...",Manhattan,6,-0.134900,0,-5.0,0.083398,0.0,46.0,1010.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1278,1278,2025-06-21 12:30:00,"POLYGON ((-73.9743296272132 40.78160532554132,...","[""I just saw someone blocking the bike lane on...",Manhattan,223,0.294327,0,28.0,0.329696,0.0,44.0,1020.0,16.0
1279,1279,2025-06-22 06:30:00,"POLYGON ((-73.9743296272132 40.78160532554132,...",['Just got a parking ticket on Columbus Avenue...,Manhattan,72,-1.557482,0,24.0,0.329696,9.0,69.0,1018.0,16.0
1280,1280,2025-07-08 09:30:00,POLYGON ((-73.96628726515767 40.58021135286303...,"[""Driving down Ocean Parkway and there's a car...",Brooklyn,27,0.974278,0,28.0,0.329696,0.0,70.0,1015.0,16.0
1281,1281,2025-07-08 09:30:00,POLYGON ((-73.96628726515767 40.58021135286303...,"[""Just got a posted parking sign violation on ...",Brooklyn,27,0.974278,0,28.0,0.329696,0.0,70.0,1015.0,16.0


In [2]:
import re

def clean(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = text.lower()
    return text

df["tweets"] = df["tweets"].apply(clean)
df

,Unnamed: 0,timestamp,geometry,tweets,Boro,Vol,congestion,congestion_level,Temperature,Condition,Wind,Humidity,Pressure,Visibility
0,0,2025-02-22 03:30:00,POLYGON ((-73.98203507773665 40.65524805627688...,"['just drove to the store and it was a breeze,...",Brooklyn,8,-1.589893,0,-6.0,0.150429,9.0,54.0,1029.0,16.0
1,1,2024-11-09 09:15:00,POLYGON ((-73.93291959036472 40.82741450154307...,"[""just headed out for a run and it's a beautif...",Manhattan,264,0.560178,0,7.0,0.329696,0.0,40.0,1027.0,16.0
2,2,2025-03-06 13:45:00,POLYGON ((-73.97178007473232 40.79434366943585...,"[""just braved the chilly morning commute, it's...",Manhattan,123,0.635957,0,11.0,0.127825,0.0,50.0,992.0,16.0
3,3,2024-04-04 15:30:00,POLYGON ((-73.98780645780917 40.67354750428635...,"['woke up to a pretty chilly grey day, 9°c out...",Brooklyn,151,0.744760,0,9.0,0.127825,6.0,52.0,996.0,16.0
4,4,2025-01-08 21:15:00,POLYGON ((-73.91101950432183 40.87001883438244...,"['just got out of my car, finally parked in a ...",Manhattan,6,-0.134900,0,-5.0,0.083398,0.0,46.0,1010.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1278,1278,2025-06-21 12:30:00,"POLYGON ((-73.9743296272132 40.78160532554132,...","[""i just saw someone blocking the bike lane on...",Manhattan,223,0.294327,0,28.0,0.329696,0.0,44.0,1020.0,16.0
1279,1279,2025-06-22 06:30:00,"POLYGON ((-73.9743296272132 40.78160532554132,...",['just got a parking ticket on columbus avenue...,Manhattan,72,-1.557482,0,24.0,0.329696,9.0,69.0,1018.0,16.0
1280,1280,2025-07-08 09:30:00,POLYGON ((-73.96628726515767 40.58021135286303...,"[""driving down ocean parkway and there's a car...",Brooklyn,27,0.974278,0,28.0,0.329696,0.0,70.0,1015.0,16.0
1281,1281,2025-07-08 09:30:00,POLYGON ((-73.96628726515767 40.58021135286303...,"[""just got a posted parking sign violation on ...",Brooklyn,27,0.974278,0,28.0,0.329696,0.0,70.0,1015.0,16.0


In [3]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')
split_idx = int(len(df) * 0.7)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()
for dataset in [train_df, test_df]:
    dataset['month'] = dataset['timestamp'].dt.month
    dataset['day'] = dataset['timestamp'].dt.day
    dataset['day_of_week'] = dataset['timestamp'].dt.dayofweek
    dataset['hour'] = dataset['timestamp'].dt.hour
    dataset['minute'] = dataset['timestamp'].dt.minute
features_times = ['month', 'day', 'day_of_week', 'hour', 'minute']
features_weather = ['Temperature','Condition', 'Wind', 'Humidity', 'Pressure', 'Visibility']
features_text = ['tweets']
X_train_times = torch.FloatTensor(train_df[features_times].values)
X_test_times = torch.FloatTensor(test_df[features_times].values)
X_train_weather = torch.FloatTensor(train_df[features_weather].values)
X_test_weather = torch.FloatTensor(test_df[features_weather].values)
y_test = torch.FloatTensor(test_df['congestion_level'].values).unsqueeze(1)
y_train = torch.FloatTensor(train_df['congestion_level'].values).unsqueeze(1)
y_train = y_train.view(-1).long()
y_test = y_test.view(-1).long()

In [4]:
!unzip encoder_model.zip

Archive:  encoder_model.zip
   creating: encoder_model/
  inflating: encoder_model/config.json  
  inflating: encoder_model/model.safetensors  
  inflating: encoder_model/tokenizer.json  
  inflating: encoder_model/tokenizer_config.json  
  inflating: encoder_model/training_args.bin  


In [5]:
from datasets import Dataset
class FusionDataset(Dataset):
    def __init__(self, time_feat, weather_data, encodings, labels):
        self.time_feat = time_feat
        self.weather_data = weather_data
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'time_feat': self.time_feat[idx],
            'weather': self.weather_data[idx],
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label': self.labels[idx]
        }


In [6]:
tokenizer = AutoTokenizer.from_pretrained("encoder_model")
train_encodings = tokenizer(
    train_df['tweets'].tolist(),
    padding=True,
    truncation=True,
    return_tensors='pt'
)

test_encodings = tokenizer(
    test_df['tweets'].tolist(),
    padding=True,
    truncation=True,
    return_tensors='pt'
)

In [7]:
time_model = torch.load("time_weights.pt", weights_only=False)
weather_model = torch.load("weather_weights.pt", weights_only=False)
text_model = AutoModel.from_pretrained("encoder_model")
time_model.eval()
time_model = time_model.cpu()
X_train_times_cpu = X_train_times.cpu()
X_test_times_cpu = X_test_times.cpu()
time_train_feat = time_model(X_train_times_cpu).detach().to(device)
time_test_feat = time_model(X_test_times_cpu).detach().to(device)


/tmp/ipykernel_2640/2405385295.py:1: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  time_model = torch.load("time_weights.pt", weights_only=False)
/tmp/ipykernel_2640/2405385295.py:2: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  weather_model = torch.load("weather_weights.pt", weights_only=False)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: encoder_model
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
from torch.utils.data import DataLoader

train_dataset = FusionDataset(
    time_train_feat,
    X_train_weather,
    train_encodings,
    y_train
)

test_dataset = FusionDataset(
    time_test_feat,
    X_test_weather,
    test_encodings,
    y_test
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [12]:
class FusionModel(nn.Module):
    def __init__(self, weather_model, text_model, num_classes=3, embed_dim=128, dropout=0.4):
        super(FusionModel, self).__init__()
        self.weather_model = weather_model
        self.text_model = text_model

        hidden_size = text_model.config.hidden_size
        self.time_proj = nn.Linear(32, embed_dim)
        self.weather_proj = nn.Linear(16, embed_dim)
        self.text_proj = nn.Linear(hidden_size, embed_dim)

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 3, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, time_feat, weather_x, input_ids, attention_mask):
        weather_feat = self.weather_model(weather_x)
        time_p = self.time_proj(time_feat)
        weather_p = self.weather_proj(weather_feat)
        text_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_feat = text_outputs.last_hidden_state[:, 0, :]  # CLS token
        text_p = self.text_proj(text_feat)
        fused = torch.cat([time_p, weather_p, text_p], dim=1)
        fused = self.dropout(fused)
        logits = self.classifier(fused)
        return logits

In [13]:
from sklearn.utils.class_weight import compute_class_weight
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 3
epochs = 20
patience = 4
max_grad_norm = 1.0
fusion_model = FusionModel(weather_model, text_model, num_classes).to(device)
y_train_np = y_train.cpu().numpy().astype(int)
classes = np.unique(y_train_np)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train_np
)

class_weights = torch.tensor(weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam([
    {'params': fusion_model.text_model.parameters(), 'lr': 2e-5},
    {'params': fusion_model.weather_model.parameters(), 'lr': 1e-4},
    {'params': fusion_model.time_proj.parameters(), 'lr': 1e-4},
    {'params': fusion_model.weather_proj.parameters(), 'lr': 1e-4},
    {'params': fusion_model.text_proj.parameters(), 'lr': 1e-4},
    {'params': fusion_model.classifier.parameters(), 'lr': 1e-4},
])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=2,
    factor=0.5
)


best_val_loss = float('inf')
counter = 0

for epoch in range(epochs):
    fusion_model.train()
    train_loss = 0

    for batch in train_loader:
        time_feat = batch['time_feat'].to(device)
        weather_x = batch['weather'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        y = batch['label'].to(device).long().view(-1)

        logits = fusion_model(time_feat, weather_x, input_ids, attention_mask)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), max_grad_norm)
        optimizer.step()

        train_loss += loss.item() * y.size(0)

    train_loss /= len(train_loader.dataset)
    fusion_model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in test_loader:
            time_feat = batch['time_feat'].to(device)
            weather_x = batch['weather'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            y = batch['label'].to(device).long().view(-1)

            logits = fusion_model(time_feat, weather_x, input_ids, attention_mask)
            loss = criterion(logits, y)

            val_loss += loss.item() * y.size(0)

    val_loss /= len(test_loader.dataset)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    scheduler.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(fusion_model.state_dict(), "best_simple_fusion_model.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered!")
            break

Epoch 1: Train Loss=6.1740, Val Loss=0.5773
Epoch 2: Train Loss=3.3220, Val Loss=0.5489
Epoch 3: Train Loss=1.5391, Val Loss=1.4966
Epoch 4: Train Loss=0.5773, Val Loss=0.9958
Epoch 5: Train Loss=0.3234, Val Loss=1.7298
Epoch 6: Train Loss=0.3044, Val Loss=1.3735
Early stopping triggered!


In [14]:
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
fusion_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        time_feat = batch['time_feat'].to(device)
        weather_x = batch['weather'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        y = batch['label'].to(device).long().view(-1)

        logits = fusion_model(time_feat, weather_x, input_ids, attention_mask)
        preds_class = torch.argmax(logits, dim=1)
        all_preds.append(preds_class.cpu())
        all_labels.append(y.cpu())

y_pred_test = torch.cat(all_preds, dim=0)
y_test_full = torch.cat(all_labels, dim=0)

f1 = f1_score(y_test_full.numpy(), y_pred_test.numpy(), average='micro')
precision = precision_score(y_test_full.numpy(), y_pred_test.numpy(), average='macro')
recall = recall_score(y_test_full.numpy(), y_pred_test.numpy(), average='macro')
cm = classification_report(y_test_full.numpy(), y_pred_test.numpy())

print("F1 Score (micro):", f1)
print("Precision (macro):", precision)
print("Recall (macro):", recall)
print("Classification Report:\n", cm)

F1 Score (micro): 0.825974025974026
Precision (macro): 0.823876742960706
Recall (macro): 0.8030351539988474
Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.89      0.88       196
           1       0.66      0.70      0.68        97
           2       0.95      0.82      0.88        92

    accuracy                           0.83       385
   macro avg       0.82      0.80      0.81       385
weighted avg       0.83      0.83      0.83       385

